# Python for Data Analysis, chapter 6: loading and saving data

The most useful examples from the chapter, rewritten for **pandas 3** and run on the files in `../samples`.

**How to use this notebook**
1. Run each cell with **Shift+Enter** and read the output before moving on.
2. Change something in each example and guess the result before running it.
3. The **Your turn** cells at the end are empty on purpose: write those yourself.

Book chapter: https://wesmckinney.com/book/accessing-data

In [16]:
import io
import json
import sqlite3
from pathlib import Path

import pandas as pd

SAMPLES = Path("../samples")   # this notebook lives in python/, the data in samples/

## 1. Reading CSV files

`pd.read_csv` is the function you will use most. To keep the examples self-contained, the CSV text is in a string, and `io.StringIO` makes the string behave like a file. With a real file, pass its path instead.

In [17]:
csv_text = """ticket_id,opened,priority,team,minutes_to_close
101,2026-11-01 08:15,high,network,45
102,2026-11-01 09:30,low,support,
103,2026-11-02 11:00,medium,support,120
104,2026-11-02 14:20,high,network,NA
105,2026-11-03 10:05,low,data,30
"""

tickets = pd.read_csv(io.StringIO(csv_text))
tickets

,ticket_id,opened,priority,team,minutes_to_close
0,101,2026-11-01 08:15,high,network,45.0
1,102,2026-11-01 09:30,low,support,NaN
2,103,2026-11-02 11:00,medium,support,120.0
3,104,2026-11-02 14:20,high,network,NaN
4,105,2026-11-03 10:05,low,data,30.0


In [18]:
tickets.dtypes

ticket_id             int64
opened                  str
priority                str
team                    str
minutes_to_close    float64
dtype: object

Notice: `opened` was read as plain text, and `minutes_to_close` became `float64` because of the empty value and `NA`. Both are common surprises. The options below fix them.

In [19]:
tickets = pd.read_csv(
    io.StringIO(csv_text),
    parse_dates=["opened"],             # read this column as datetimes
    dtype={"minutes_to_close": "Int64"},  # whole numbers that allow missing values
    na_values=["NA", ""],                # which texts count as missing
)
tickets.dtypes

ticket_id                    int64
opened              datetime64[us]
priority                       str
team                           str
minutes_to_close             Int64
dtype: object

Other options worth knowing, each with a small example:

In [20]:
# A file with no header row: give the column names yourself
no_header = """1,dana,82
2,avi,45"""
pd.read_csv(io.StringIO(no_header), header=None, names=["id", "name", "score"])

,id,name,score
0,1,dana,82
1,2,avi,45


In [21]:
# A different separator, such as ; or a tab ("\t")
semicolon = """id;name
1;dana
2;avi"""
pd.read_csv(io.StringIO(semicolon), sep=";")

,id,name
0,1,dana
1,2,avi


In [22]:
# Only some columns, and only the first rows: useful to peek at a huge file
pd.read_csv(io.StringIO(csv_text), usecols=["ticket_id", "priority"], nrows=3)

,ticket_id,priority
0,101,high
1,102,low
2,103,medium


In [23]:
# Big files: read in pieces instead of all at once
for chunk in pd.read_csv(io.StringIO(csv_text), chunksize=2):
    print(len(chunk), "rows in this chunk")

2 rows in this chunk
2 rows in this chunk
1 rows in this chunk


## 2. Writing CSV files

`index=False` leaves out pandas' row numbers, which you almost never want in a file.

In [24]:
tickets.to_csv("tickets_out.csv", index=False)
print(Path("tickets_out.csv").read_text())

ticket_id,opened,priority,team,minutes_to_close
101,2026-11-01 08:15:00,high,network,45
102,2026-11-01 09:30:00,low,support,
103,2026-11-02 11:00:00,medium,support,120
104,2026-11-02 14:20:00,high,network,
105,2026-11-03 10:05:00,low,data,30



## 3. JSON: flat lists of records

`bus_rides.json` is a list of objects, one per ride. That shape maps straight onto a table: each object is a row, each key a column.

In [25]:
rides = pd.read_json(SAMPLES / "bus_rides.json")
print(rides.shape)
rides[["id", "scheduled_start_time", "duration_minutes", "siri_route__operator_ref"]].head()

(5, 29)


,id,scheduled_start_time,duration_minutes,siri_route__operator_ref
0,27418192,2023-01-01 06:30:00+00:00,24,3
1,27418193,2023-01-01 06:15:00+00:00,20,3
2,27418194,2023-01-01 06:45:00+00:00,49,3
3,27418195,2023-01-01 05:45:00+00:00,6,3
4,27418196,2023-01-01 06:15:00+00:00,36,3


## 4. JSON: nested data and `json_normalize`

`github_incidents.json` is nested: an outer object holding a list of incidents, and each incident holding its own list of updates. `pd.read_json` can't flatten that well. Load it with the `json` module and flatten it with `pd.json_normalize`. This is the most useful tool in the chapter for API data.

In [26]:
with open(SAMPLES / "github_incidents.json") as f:
    raw = json.load(f)

raw.keys()

dict_keys(['page', 'incidents'])

In [27]:
# One row per incident
incidents = pd.json_normalize(raw["incidents"])
incidents[["id", "name", "status", "impact", "created_at", "resolved_at"]].head()

,id,name,status,impact,created_at,resolved_at
0,hnf21m9jkl3c,Incident with Git Operations and Pull Requests,resolved,critical,2026-11-10T16:10:52.174Z,2026-11-10T17:55:03.552Z
1,6jm5k16vyhfg,Degraded availability for GPT-5.3 Codex and GP...,resolved,major,2026-11-09T23:16:32.233Z,2026-11-10T00:05:52.345Z
2,tp8cghg7tlc4,Incident with Actions and Checks API,resolved,critical,2026-11-09T18:12:37.242Z,2026-11-09T18:59:32.565Z
3,xxb3l2bm76dr,Incident with Webhooks and Actions,resolved,major,2026-11-09T10:00:35.232Z,2026-11-09T11:50:35.774Z
4,c9px7cw2ghrs,Increased timeouts due to surge in anonymous u...,resolved,minor,2026-11-09T05:59:07.851Z,2026-11-09T09:09:56.174Z


In [28]:
# One row per update, keeping the incident it belongs to
updates = pd.json_normalize(
    raw["incidents"],
    record_path="incident_updates",           # the inner list to expand
    meta=["id", "name"],                       # fields to copy from the parent
    meta_prefix="parent_",                     # updates already have an "incident_id" field
)
updates[["parent_id", "parent_name", "status", "created_at"]].head()

,parent_id,parent_name,status,created_at
0,hnf21m9jkl3c,Incident with Git Operations and Pull Requests,resolved,2026-11-10T17:55:03.552Z
1,hnf21m9jkl3c,Incident with Git Operations and Pull Requests,investigating,2026-11-10T17:46:33.967Z
2,hnf21m9jkl3c,Incident with Git Operations and Pull Requests,investigating,2026-11-10T17:28:17.898Z
3,hnf21m9jkl3c,Incident with Git Operations and Pull Requests,investigating,2026-11-10T17:12:54.802Z
4,hnf21m9jkl3c,Incident with Git Operations and Pull Requests,investigating,2026-11-10T17:07:48.682Z


A first real question, using what you learned in chapter 5: how long did each incident take to resolve?

In [29]:
incidents["created_at"] = pd.to_datetime(incidents["created_at"])
incidents["resolved_at"] = pd.to_datetime(incidents["resolved_at"])
incidents["hours_to_resolve"] = (
    (incidents["resolved_at"] - incidents["created_at"]).dt.total_seconds() / 3600
).round(1)

incidents.sort_values("hours_to_resolve", ascending=False)[["name", "impact", "hours_to_resolve"]].head()

,name,impact,hours_to_resolve
42,Incident across several services,minor,18.7
25,Project updates delayed,minor,11.0
45,Degradation with Gemini 3.8 Flash,major,10.5
31,"Incident with Actions, Pull Requests and Issues",minor,7.9
27,Delays for Pull Requests and Webhook deliveries,critical,4.5


## 5. Parquet: the format data engineers store data in

CSV stores everything as text and loses the types. **Parquet** is a binary format that keeps the types, compresses well and lets tools read only the columns they need. Your projects will store raw data as Parquet.

It needs one extra package. In the Ubuntu terminal, with your `.venv` active, run `pip install pyarrow` once.

In [30]:
tickets.to_parquet("tickets.parquet", index=False)
back = pd.read_parquet("tickets.parquet")
back.dtypes   # the types survived, unlike with CSV

ticket_id                    int64
opened              datetime64[us]
priority                       str
team                           str
minutes_to_close             Int64
dtype: object

## 6. Calling a web API with `requests`

The pattern every ingestion script uses: request with a **timeout**, fail loudly on a bad status, then parse the JSON. This cell needs an internet connection.

In [31]:
import requests

url = "https://www.githubstatus.com/api/v2/incidents.json"
resp = requests.get(url, timeout=10)   # never call an API without a timeout
resp.raise_for_status()                # turns HTTP errors (404, 500) into a Python error

live = pd.json_normalize(resp.json()["incidents"])
live[["name", "status", "impact", "created_at"]].head()

,name,status,impact,created_at
0,LFS degradation for some customers,resolved,minor,2026-11-25T23:06:54.830Z
1,Investigating degraded experiences across GitH...,resolved,critical,2026-11-25T15:55:19.609Z
2,MAI Code 2 Degredation,resolved,minor,2026-11-25T11:58:36.151Z
3,Disruption with Code Completions and Next Edit...,resolved,critical,2026-11-25T00:08:17.974Z
4,Degraded Availability for MAI Code 2 in Copilot,resolved,minor,2026-11-24T13:44:07.213Z


## 7. Databases: SQL and pandas together

Python ships with **SQLite**, a database stored in one file. `to_sql` writes a DataFrame into a table and `pd.read_sql` runs a query and returns a DataFrame, so all your SQL practice applies here.

In [32]:
con = sqlite3.connect("practice.db")

incidents[["id", "name", "impact", "hours_to_resolve"]].to_sql(
    "incidents", con, if_exists="replace", index=False
)

pd.read_sql(
    """
    select impact, count(*) as incidents, round(avg(hours_to_resolve), 1) as avg_hours
    from incidents
    group by impact
    order by avg_hours desc
    """,
    con,
)

,impact,incidents,avg_hours
0,minor,29,2.5
1,major,8,2.5
2,critical,12,1.8
3,none,1,0.0


In [33]:
con.close()

# Remove the practice files so they don't end up in Git
for name in ["tickets_out.csv", "tickets.parquet", "practice.db"]:
    Path(name).unlink(missing_ok=True)

## Your turn

Write these yourself, without copying from above. Each takes a few lines.

1. Load `bus_rides.json` and find the average `duration_minutes` per `siri_route__operator_ref`. (Chapter 5: `groupby` or `value_counts`.)
2. From `updates`, count how many updates each incident had, and show the incident with the most.
3. Write the `incidents` table to a CSV and read it back. Which column types changed? Then do the same with Parquet and compare.
4. Call `https://www.githubstatus.com/api/v2/components.json` with `requests` and make a table of component `name` and `status`.

In [38]:
rides = pd.read_json(SAMPLES / "bus_rides.json")
rides["duration_minutes"].groupby(rides["siri_route__operator_ref"]).mean()

siri_route__operator_ref
3    27.0
Name: duration_minutes, dtype: float64

In [49]:
count = updates.groupby(updates["incident_id"]).size()
print(count)
count.max()

incident_id
0rn90wk115q9     6
13b6lyl8d7bz     6
169pg72gg9jr     4
1dk955gg3bvz     7
2dpbcq5j165n    10
2kxnqcxcrr7f     3
3q1yb5m7ltvb    12
6jm5k16vyhfg     7
6yk962mhlr3h     1
71y513kjvbzh     8
8tzbpcnl75jp     6
8zc63m64hy36    16
93njd414rpbx    13
bk7zgdcq7s9t     3
c8466lzvnmsv     4
c9px7cw2ghrs    10
cjwszcf57bff     9
db4x8hh362t8     5
djlmxz2zd0j7    20
dnslv7xdl4fj     1
dqn46wtvbdzv     1
f1vvv30224wd     4
f6yrxnz5f7bs     4
h02n9572mx9y     4
hnf21m9jkl3c     9
khv5nj9qy2kk     4
kzh5j0mt7qwc     3
l7ycmglmykgg     8
lyyfm7j4pkgl     6
nlxnbqnkdzdl     5
nms1dpzmh9d9    15
ntrfd2jqldn6     5
nvy2q96mcmyg     3
nxy2l0ssxm10     8
pfbmpxxfq1cy     8
pwxsjgnvrk6b     6
pyynvrm1vskw     7
q29p76qtnkyk     7
qpfv5p86dmrl     5
qwdwmtqghpk5     7
rn76zdy367m2     6
rxlj89wpk7px     7
smtdswd1h2w4     9
t89g2q1800fr     4
tc28c0wzrnpz     6
tp8cghg7tlc4     7
vlxrztp88101     4
wq8305lppdc2    18
xr3myzj7ms51     9
xxb3l2bm76dr     8
dtype: int64


np.int64(20)

In [50]:
incidents.to_csv("incidents.csv", index=False)
from_csv = pd.read_csv("incidents.csv")
compare = pd.DataFrame({
    "original": incidents.dtypes,
    "after_csv": from_csv.dtypes,
})
compare

,original,after_csv
id,str,str
name,str,str
status,str,str
created_at,"datetime64[us, UTC]",str
updated_at,str,str
monitoring_at,str,str
resolved_at,"datetime64[us, UTC]",str
impact,str,str
shortlink,str,str
started_at,str,str


In [52]:
incidents.to_parquet("incidents.csv", index=False)
from_parquet = pd.read_parquet("incidents.csv")
compare = pd.DataFrame({
    "original": incidents.dtypes,
    "after_parquet": from_parquet.dtypes,
})
compare

,original,after_parquet
id,str,str
name,str,str
status,str,str
created_at,"datetime64[us, UTC]","datetime64[us, UTC]"
updated_at,str,str
monitoring_at,str,str
resolved_at,"datetime64[us, UTC]","datetime64[us, UTC]"
impact,str,str
shortlink,str,str
started_at,str,str


In [58]:
url = "https://www.githubstatus.com/api/v2/components.json"
resp = requests.get(url, timeout=10)   # never call an API without a timeout
resp.raise_for_status()                # turns HTTP errors (404, 500) into a Python error

live = pd.json_normalize(resp.json()["components"])
live[["name", "status"]]

,name,status
0,Git Operations,operational
1,Webhooks,operational
2,Visit www.githubstatus.com for more information,operational
3,API Requests,operational
4,Issues,operational
5,Pull Requests,operational
6,Actions,operational
7,Packages,operational
8,Pages,operational
9,Copilot,operational


## Notes in my own words

- **CSV vs Parquet:** CSV saves everything as text, so types are lost: dates came back
  as text in exercise 3. Parquet keeps each column's type, is smaller, and lets tools read
  only some columns. CSV is for sharing with people; Parquet is for storing pipeline data.
- **json_normalize:** turns nested JSON (lists inside objects) into a flat table.
  `record_path` picks the inner list to expand; `meta` copies fields from the parent,
  like the incident ID on each update.
- **Every API call needs a timeout:** without one, if the server never answers, the
  script waits forever and a scheduled pipeline hangs silently. `raise_for_status()`
  makes HTTP errors fail loudly instead of passing bad data along.
- **No loops in pandas:** use `groupby(...).mean()`, `.size()`, `.idxmax()` instead
  of looping over rows. `.size()` counts rows per group; `.count()` counts non-empty
  values in every column.